# Baseline Model & Forecasting Experiments

## Objective

Establish a seasonal-naive baseline (this week = same week last year) as the minimum bar for any forecasting model. Per the time-series analysis, the series has weak trend and concentrated holiday seasonality, which is exactly what a seasonal-naive baseline captures without any fitting — this determines whether more complex models actually add value.

In [3]:
import pandas as pd
import numpy as np

model_data = pd.read_csv("../data/processed/store_week_features.csv", parse_dates=["Date"])

dates_sorted = sorted(model_data["Date"].unique())
n = len(dates_sorted)
train_end = dates_sorted[int(n * 0.70)]
val_end = dates_sorted[int(n * 0.85)]

train_set = model_data[model_data["Date"] <= train_end].copy()
val_set = model_data[(model_data["Date"] > train_end) & (model_data["Date"] <= val_end)].copy()
test_set = model_data[model_data["Date"] > val_end].copy()

print(len(train_set), len(val_set), len(test_set))

2880 630 585


## Seasonal-Naive Baseline

**Decision:** Predict Weekly_Sales for each store-week using lag_52 (same store, same week, one year earlier) — already computed and leakage-checked in feature engineering. No fitting required; this is a pure lookup.

In [6]:
val_set["baseline_pred"] = val_set["lag_52"]

print(val_set[["Store","Date","Weekly_Sales","baseline_pred"]].head(10))
print()
print("Any missing baseline predictions in val_set:", val_set["baseline_pred"].isnull().sum())

    Store       Date  Weekly_Sales  baseline_pred
64      1 2012-04-27    1468928.37     1455090.69
65      1 2012-05-04    1684519.99     1629391.28
66      1 2012-05-11    1611096.05     1604775.58
67      1 2012-05-18    1595901.87     1428218.27
68      1 2012-05-25    1555444.55     1466046.67
69      1 2012-06-01    1624477.58     1635078.41
70      1 2012-06-08    1697230.96     1588948.32
71      1 2012-06-15    1630607.00     1532114.86
72      1 2012-06-22    1527845.81     1438830.15
73      1 2012-06-29    1540421.49     1488538.09

Any missing baseline predictions in val_set: 0


## Evaluation Metrics

**Decision:** Use MAE, RMSE, and WAPE.

**Why:** MAE gives an average error in the same units as sales (easy to communicate to a business stakeholder). RMSE penalizes large misses more heavily, which matters here since we observed extreme spikes (holiday weeks) that a model could badly miss. WAPE (weighted absolute percentage error = sum of absolute errors / sum of actuals) is used instead of plain MAPE because MAPE is undefined/unstable when actual sales are near zero, and gives a normalized error that's comparable across stores of very different sizes — useful since we found Type A/B/C stores differ by an order of magnitude in scale.

In [9]:
def evaluate(y_true, y_pred, label=""):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))
    print(f"{label} MAE: {mae:,.2f}")
    print(f"{label} RMSE: {rmse:,.2f}")
    print(f"{label} WAPE: {wape:.4f}")
    return {"MAE": mae, "RMSE": rmse, "WAPE": wape}

baseline_metrics = evaluate(val_set["Weekly_Sales"], val_set["baseline_pred"], label="Seasonal-Naive Baseline")

Seasonal-Naive Baseline MAE: 54,807.97
Seasonal-Naive Baseline RMSE: 78,012.51
Seasonal-Naive Baseline WAPE: 0.0523


In [15]:
val_set["abs_error"] = np.abs(val_set["Weekly_Sales"] - val_set["baseline_pred"])

type_error = val_set.groupby("Type").apply(
    lambda d: pd.Series({
        "WAPE": d["abs_error"].sum() / d["Weekly_Sales"].abs().sum(),
        "MAE": d["abs_error"].mean()
    })
)
print(type_error)

          WAPE           MAE
Type                        
A     0.052218  72039.665130
B     0.051156  41893.064874
C     0.058122  28217.287738


/var/folders/l0/_xc3x7m10hxf3hjl2cj6gz2c0000gn/T/ipykernel_7343/1856070898.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  type_error = val_set.groupby("Type").apply(


### Findings: Seasonal-Naive Baseline

**Observed:** The seasonal-naive baseline (predict this week = same week last year) achieves MAE 54,807.97, RMSE 78,012.51, and WAPE 0.0523 on the validation set — meaning predictions are off by about 5.2% of total sales on average, without any model fitting at all. This is a strong baseline, consistent with the concentrated, calendar-recurring seasonality found earlier.

By store Type, WAPE is 0.0522 (A), 0.0512 (B), and 0.0581 (C). This is a genuine surprise: Type C has the worst baseline error despite being the smallest and, per the earlier store-level comparison, the least volatile Type by average coefficient of variation. The likely explanation is sample size — only 6 Type C stores exist, so the group average is more sensitive to one or two individual stores with an atypical year-over-year pattern, rather than Type C being intrinsically harder to forecast as a category.

**Decision:** Any real forecasting model must beat WAPE 0.0523 overall to be worth using over the free seasonal-naive lookup. Store Type C's baseline performance should be checked at the individual-store level (not just the group average) once we get to store-level error analysis, since a 6-store group average can be misleading.

**Why:** Without this baseline, a model with a WAPE of, say, 0.07 might look "reasonable" in isolation but would actually be worse than doing nothing. This number is the actual bar, not an assumed one.

## Model 1: Holt-Winters Exponential Smoothing

**Decision:** Fit a separate Holt-Winters model per store (additive trend, additive seasonality, 52-week seasonal period), trained on train_set only, evaluated on val_set.

**Why:** The time-series analysis found weak trend and strong, recurring annual seasonality — Holt-Winters is designed exactly for this combination and is a standard, explainable next step up from a naive lookup. Per-store models are used rather than one global model, since the store-level comparison found stores differ substantially in scale (Type A/B/C), and forcing one seasonal pattern onto stores of very different sizes would be a poor fit.

In [23]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings("ignore")

hw_predictions = []
hw_failures = []

for store_id in sorted(train_set["Store"].unique()):
    train_store = train_set[train_set["Store"] == store_id].sort_values("Date")
    val_store = val_set[val_set["Store"] == store_id].sort_values("Date")

    if len(val_store) == 0:
        continue

    try:
        model = ExponentialSmoothing(
            train_store["Weekly_Sales"],
            trend="add",
            seasonal="add",
            seasonal_periods=52,
            initialization_method="estimated"
        ).fit()

        forecast = model.forecast(len(val_store))
        val_store = val_store.copy()
        val_store["hw_pred"] = forecast.values
        hw_predictions.append(val_store)
    except Exception as e:
        hw_failures.append((store_id, str(e)))

print("Stores succeeded:", len(hw_predictions))
print("Stores failed:", len(hw_failures))
for store_id, err in hw_failures[:5]:
    print(f"Store {store_id}: {err}")

if hw_predictions:
    hw_results = pd.concat(hw_predictions)
    print("Total val rows covered:", len(hw_results))
else:
    print("No stores succeeded — Holt-Winters is not viable with this train window length.")

Stores succeeded: 0
Stores failed: 45
Store 1: Cannot compute initial seasonals using heuristic method with less than two full seasonal cycles in the data.
Store 2: Cannot compute initial seasonals using heuristic method with less than two full seasonal cycles in the data.
Store 3: Cannot compute initial seasonals using heuristic method with less than two full seasonal cycles in the data.
Store 4: Cannot compute initial seasonals using heuristic method with less than two full seasonal cycles in the data.
Store 5: Cannot compute initial seasonals using heuristic method with less than two full seasonal cycles in the data.
No stores succeeded — Holt-Winters is not viable with this train window length.


### Findings: Holt-Winters Exponential Smoothing

**Observed:** Fitting a per-store Holt-Winters model (additive trend, additive seasonality, 52-week seasonal period) failed for all 45 stores, both with the default heuristic initialization and with `initialization_method="estimated"`. The error is consistent: the model requires at least two full seasonal cycles (104 weeks) of history to estimate a 52-week seasonal pattern, but train_set only covers ~63 weeks (2011-02-04 to 2012-04-20) per store after the chronological split and lag-52 row removal.

**Decision:** Do not use Holt-Winters with 52-week seasonality for this project. Move to a tree-based regression model instead, which can use the lag_52 feature (already computed and leakage-checked) to capture year-over-year seasonality without needing a full two-cycle fit — it's a fundamentally different way of encoding the same seasonal information we already validated exists.

**Why:** Fabricating a workaround (e.g., shortening the seasonal period to fit, or fitting on a shorter cycle) would produce a technically-running but conceptually wrong model — a 52-week seasonal pattern fit on less than one full cycle isn't a meaningful seasonal decomposition, it would just be curve-fitting noise. This is a genuine data-availability constraint worth stating plainly rather than hiding: the chronological split (needed to avoid leakage) combined with the lag-52 requirement (needed for the seasonal-naive baseline and future features) leaves too short a window for this particular statistical method.

## Model 2: Gradient-Boosted Trees

**Decision:** Train a single global LightGBM/GradientBoosting regression model across all stores, using lag_1, lag_2, lag_52, rolling_mean_4, rolling_std_4, WeekOfYear, Month, HolidayName, Type, Size as features.

**Why:** Holt-Winters per-store wasn't viable given the available history. A tree-based model can pool information across all 45 stores (more effective training data per model than fitting 45 separate small models) while still letting Type/Size/HolidayName features capture store-specific and calendar-specific behavior. lag_52 lets it learn a year-over-year seasonal signal directly as a feature, which is the same information the seasonal-naive baseline uses, but combined with everything else.

In [32]:
try:
    import lightgbm as lgb
    HAS_LGB = True
    print("LightGBM available")
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    HAS_LGB = False
    print("LightGBM not available, using sklearn GradientBoostingRegressor")

LightGBM not available, using sklearn GradientBoostingRegressor


In [34]:
feature_cols = ["lag_1", "lag_2", "lag_52", "rolling_mean_4", "rolling_std_4",
                "WeekOfYear", "Month", "HolidayName", "Type", "Size"]

train_encoded = pd.get_dummies(train_set[feature_cols + ["Weekly_Sales"]], columns=["HolidayName", "Type"])
val_encoded = pd.get_dummies(val_set[feature_cols + ["Weekly_Sales"]], columns=["HolidayName", "Type"])

# Align columns in case a category is missing in one split
train_encoded, val_encoded = train_encoded.align(val_encoded, join="left", axis=1, fill_value=0)

X_train = train_encoded.drop(columns=["Weekly_Sales"])
y_train = train_encoded["Weekly_Sales"]
X_val = val_encoded.drop(columns=["Weekly_Sales"])
y_val = val_encoded["Weekly_Sales"]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Any NaNs in X_train:", X_train.isnull().sum().sum())
print("Any NaNs in X_val:", X_val.isnull().sum().sum())

X_train shape: (2880, 15)
X_val shape: (630, 15)
Any NaNs in X_train: 0
Any NaNs in X_val: 0


In [36]:
if HAS_LGB:
    model_gb = lgb.LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1)
else:
    model_gb = GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)

model_gb.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [38]:
gb_pred = model_gb.predict(X_val)

gb_metrics = evaluate(y_val, gb_pred, label="Gradient Boosting")

print()
print("For comparison — Seasonal-Naive Baseline WAPE was: 0.0523")

Gradient Boosting MAE: 38,857.63
Gradient Boosting RMSE: 58,645.57
Gradient Boosting WAPE: 0.0371

For comparison — Seasonal-Naive Baseline WAPE was: 0.0523


In [40]:
importances = pd.Series(model_gb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(10))

lag_52                      0.873592
rolling_mean_4              0.097832
lag_1                       0.016680
lag_2                       0.003170
WeekOfYear                  0.003063
Size                        0.002602
rolling_std_4               0.001483
Month                       0.000901
HolidayName_Thanksgiving    0.000351
HolidayName_SuperBowl       0.000173
dtype: float64


### Findings: Gradient Boosting

**Observed:** The gradient boosting model achieves MAE 38,857.63, RMSE 58,645.57, and WAPE 0.0371 on the validation set — a genuine improvement over the seasonal-naive baseline's WAPE of 0.0523 (roughly a 29% relative reduction in error). Feature importance is dominated by lag_52 (0.874), followed distantly by rolling_mean_4 (0.098) and lag_1 (0.017); calendar and holiday-identity features contribute very little individually (each under 0.01).

**Decision:** Gradient boosting is the stronger candidate so far. It essentially learns a refined version of the seasonal-naive logic (still leaning heavily on year-ago sales) but corrects it using recent short-term momentum (rolling_mean_4) and, to a lesser extent, recent weeks (lag_1) and store scale.

**Why:** The feature importance ranking is itself evidence the model is doing something sensible rather than overfitting to noise